# Sector-Level Hashtag Association Mining


# Hashtag Association Mining

This notebook treats each post as one transaction and each hashtag as an item.

This variant runs association mining per `sector`.

In [1]:

import re
from pathlib import Path

import numpy as np
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 180)


In [2]:

DATA_PATH = Path('synthetic_generator') / 'synthetic_social_media_posts_with_kpis.csv'
OUTPUT_DIR = Path('notebooks') / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)
print('Rows:', len(df))
print('Columns:', list(df.columns))


Rows: 1000
Columns: ['business_name', 'sector', 'followers_count', 'post_date', 'posting_hour', 'day_of_week', 'month', 'post_type', 'caption_text', 'caption_length', 'hashtags_count', 'emoji_count', 'likes_count', 'comments_count', 'views_count', 'language', 'CTA_present', 'promo_post', 'discount_percent', 'mentions_location', 'religious_theme', 'patriotic_theme', 'arabic_dialect_style', 'engagement_count', 'engagement_rate_by_followers', 'like_rate_by_views', 'comment_rate_by_likes', 'views_per_follower', 'is_high_engagement', 'posting_time_category', 'caption_length_category', 'hashtag_intensity', 'content_localization_score']


In [3]:

# Regex captures hashtags in Arabic/English, digits, and underscore.
HASHTAG_RE = re.compile(r'(?<!\w)#([\w\u0600-\u06FF]+)', flags=re.UNICODE)

def extract_hashtags(text):
    if pd.isna(text):
        return []
    tags = [f"#{m.lower()}" for m in HASHTAG_RE.findall(str(text))]
    # de-duplicate within each post transaction
    return sorted(set(tags))

df['hashtags'] = df['caption_text'].apply(extract_hashtags)
df['n_tags_extracted'] = df['hashtags'].str.len()

print('Posts with >=1 hashtag:', int((df['n_tags_extracted'] > 0).sum()))
print('Unique hashtags:', len(set(t for tags in df['hashtags'] for t in tags)))

df[['caption_text', 'hashtags']].head(5)


Posts with >=1 hashtag: 680
Unique hashtags: 16


,caption_text,hashtags
0,صحتك بتهمنا Swipe وشوفوا الباقي. موجودين في Rafah today. منتج فلسطيني زورونا وخلينا نجهزلك طلبك. #تعليم #مخبز,"[#تعليم, #مخبز]"
1,اليوم عنا أكل طيب فيديو جديد. احنا جاهزين احجز الآن وخليها علينا #عيادة #تعليم,"[#تعليم, #عيادة]"
2,صحتكم أولويتنا سوايب وشوفوا التفاصيل. شو رأيكم؟ الكمية محدودة 50% خصم لفترة محدودة زورونا بفرعنا في Rafah ابعتولنا D...,[#عروض]
3,استنونا بدورة جديدة شوفوا الريل. زورونا بفرعنا في Tulkarm كل عام وأنتم بخير ✅ #تعليم #قهوة,"[#تعليم, #قهوة]"
4,أجواء ولا أروع صورة جديدة. أهلا بالناس الحلوة منتج فلسطيني #عيادة #مطاعم,"[#عيادة, #مطاعم]"


In [4]:

sector_results = []
sector_rules_top = []

for sector, g in df.groupby('sector', dropna=False):
    g = g.copy()
    g = g[g['n_tags_extracted'] > 0]
    tx = g['hashtags'].tolist()

    if len(tx) < 25:
        sector_results.append({
            'sector': sector,
            'n_transactions': len(tx),
            'n_unique_hashtags': len(set(t for tags in tx for t in tags)) if len(tx) else 0,
            'n_rules': 0,
            'note': 'Skipped: too few hashtagged posts'
        })
        continue

    te = TransactionEncoder()
    te_arr = te.fit(tx).transform(tx)
    onehot = pd.DataFrame(te_arr, columns=te.columns_)

    # Slightly higher support inside sector to avoid noise.
    freq = apriori(onehot, min_support=0.02, use_colnames=True)
    if freq.empty:
        sector_results.append({
            'sector': sector,
            'n_transactions': len(tx),
            'n_unique_hashtags': onehot.shape[1],
            'n_rules': 0,
            'note': 'No frequent itemsets at threshold'
        })
        continue

    rules = association_rules(freq, metric='confidence', min_threshold=0.25)
    if rules.empty:
        sector_results.append({
            'sector': sector,
            'n_transactions': len(tx),
            'n_unique_hashtags': onehot.shape[1],
            'n_rules': 0,
            'note': 'No rules at threshold'
        })
        continue

    rules = rules.sort_values(['lift', 'confidence', 'support'], ascending=[False, False, False]).reset_index(drop=True)
    rules['sector'] = sector
    rules['antecedents'] = rules['antecedents'].apply(lambda x: ', '.join(sorted(list(x))))
    rules['consequents'] = rules['consequents'].apply(lambda x: ', '.join(sorted(list(x))))

    best = rules.iloc[0]
    sector_results.append({
        'sector': sector,
        'n_transactions': len(tx),
        'n_unique_hashtags': onehot.shape[1],
        'n_rules': len(rules),
        'best_antecedent': best['antecedents'],
        'best_consequent': best['consequents'],
        'best_support': float(best['support']),
        'best_confidence': float(best['confidence']),
        'best_lift': float(best['lift']),
    })

    sector_rules_top.append(rules.head(10))

sector_summary = pd.DataFrame(sector_results).sort_values(['n_rules', 'n_transactions'], ascending=[False, False])
sector_summary.to_csv(OUTPUT_DIR / 'sector_hashtag_summary.csv', index=False)

if sector_rules_top:
    pd.concat(sector_rules_top, ignore_index=True).to_csv(OUTPUT_DIR / 'sector_hashtag_top_rules.csv', index=False)

sector_summary


,sector,n_transactions,n_unique_hashtags,n_rules,best_antecedent,best_consequent,best_support,best_confidence,best_lift
1,beauty_salon,48,16,344,"#supportlocal, #جيم","#palestine, #عيادة",0.020833,1.000000,48.000000
3,clinic,68,14,134,"#palestine, #مطاعم","#تسوق, #عيادة",0.029412,1.000000,13.600000
2,cafe,70,12,97,"#الكترونيات, #تسوق","#عيادة, #قهوة",0.028571,1.000000,17.500000
8,restaurant,78,16,90,"#الكترونيات, #عروض","#ramallah, #مطاعم",0.025641,1.000000,7.090909
7,pharmacy,72,14,89,"#nablus, #عروض","#palestine, #عيادة",0.027778,0.400000,14.400000
4,education_center,67,14,60,"#الكترونيات, #عروض",#تعليم,0.044776,0.750000,4.568182
5,electronics_store,75,12,50,"#عيادة, #مطاعم",#صيدلية,0.026667,0.666667,7.142857
9,store,86,14,39,"#supportlocal, #تسوق",#hebron,0.023256,1.000000,7.818182
6,gym,54,12,36,"#palestine, #صيدلية",#مخبز,0.037037,1.000000,9.000000
0,bakery,62,12,22,#مخبز,#الكترونيات,0.032258,0.400000,3.542857
